# مختبر اليوم الثاني — تجهيز خصائص عملاء منافذ في خط معالجة

> هذا الدفتر مبني مباشرة من مواصفات مختبرات منافذ المعتمدة للدورة.


In [ ]:
from pathlib import Path

# يعمل محليًا داخل المستودع أو عند وضع الحزمة في مجلد مستقل
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data/raw"), Path("../../data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])
CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"
print("DATA_DIR:", DATA_DIR.resolve())


## هدف المختبر

يبني الطالب خط معالجة يعالج النقص والخصائص العددية والفئوية بطريقة قابلة للتكرار، من دون أن يتعلم أي قيمة من بيانات الاختبار.

## السيناريو والبيانات

يستمر الطالب على جدول العملاء نفسه وعلى الخصائص الآمنة التي حددها في اليوم الأول. يستفيد المختبر عمدًا من النقص الواقعي في التقييم المتوسط وآخر عرض ترويجي. لا يحذف الطلاب هذين العمودين؛ الهدف هو تعلم معالجة النقص بصورة منظمة.

| مجموعة الخصائص | الأعمدة | المعالجة المطلوبة | سبب الاختيار |
|---|---|---|---|
| عددية | `tenure_months`، `orders_per_month`، `avg_basket_sar`، `days_since_last_order`، `distinct_categories`، `promo_usage_rate`، `avg_rating` | وسيط ثم تحجيم، مع إضافة مؤشر نقص | تحتوي `avg_rating` على نقص ذي معنى محتمل للعميل الجديد |
| فئوية | `city`، `device`، `payment_method`، `last_promo_used` | تعويض قيمة «غير معروف» ثم ترميز أحادي | الفئة اسم لا مقدار، وغياب آخر عرض قد يحمل معنى |
| ترتيبية | `city_tier` | تعامل كرقم ترتيبي في هذا المختبر | الترتيب بين 1 و2 و3 موجود فعليًا في وصف الحزمة |
| مستبعدة | `customer_id`، `signup_date`، `snapshot_date`، أعمدة التسريب، والهدف | لا تدخل `X` | معرّف أو تاريخ لم نشتق منه خاصية اليوم أو معلومات مستقبلية |

## خطوات التنفيذ بالتسلسل

1. أعد قراءة الملف أو استخدم متغير `customers` من اليوم الأول.
2. عرّف `X` باستخدام `safe_features` وحدها، وعرّف `y` من `churned_30d`.
3. قسم البيانات إلى تدريب واختبار بنسبة 80/20 مع `stratify=y` و`random_state=42`.
4. عرّف قائمة الخصائص العددية وقائمة الخصائص الفئوية.
5. ابنِ مسارًا عدديًا: تعويض بالوسيط، مؤشر نقص، ثم تحجيم.
6. ابنِ مسارًا فئويًا: تعويض بقيمة ثابتة «غير معروف»، ثم ترميز أحادي.
7. اجمع المسارين داخل `ColumnTransformer` باسم `preprocessor`.
8. شغّل `fit_transform` على تدريب صغير فقط للتحقق من أن الخط يعمل، ولا تدرب نموذجًا في هذا اليوم.

## ما يكتبه أو يشغله الطالب


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = customers[safe_features]
y = customers["churned_30d"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = [
    "city_tier", "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating"
]
categorical_features = [
    "city", "device", "payment_method", "last_promo_used"
]

numeric_pipeline = Pipeline([
    ("fill", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("fill", SimpleImputer(strategy="constant", fill_value="غير_معروف")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numbers", numeric_pipeline, numeric_features),
    ("categories", categorical_pipeline, categorical_features)
])

prepared_train = preprocessor.fit_transform(X_train)
prepared_test = preprocessor.transform(X_test)
print(prepared_train.shape, prepared_test.shape)


## النتيجة المتوقعة

ينتج متغير `preprocessor` ومصفوفتا خصائص مجهزتان للتدريب والاختبار، مع عدد صفوف يطابق صفوف التدريب والاختبار. قد يزيد عدد الأعمدة بعد الترميز وإضافة مؤشر النقص؛ هذا طبيعي. لا يجب أن يضم أي ناتج عمودًا من `leak_columns`.

## المهارات التي يراجعها الطالب

يفصل الطالب التدريب عن الاختبار، ويعالج القيم الناقصة، ويستخدم مؤشر النقص، ويرمز الفئات، ويحجم الأرقام، ويبني `Pipeline` و`ColumnTransformer` دون معالجة يدوية مختلفة لكل مجموعة بيانات.

## شرط التسليم قبل مغادرة اليوم

يعرض الطالب كتلتي الكود للمسارين وقائمتي الأعمدة. يشرح في جملة واحدة لماذا لا يحسب الوسيط من كامل الجدول قبل التقسيم.

---
